## Preparação

In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
import sys
from pathlib import Path

AGENTS_DIR = Path.cwd().parent / "agents"
sys.path.append(str(AGENTS_DIR))

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [ ]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

## RAG

Lookup table for FAQ documents

RAG

In [ ]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
genai_client = genai.Client()

MODEL = "gemini-3.5-flash-lite"

In [ ]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=genai_client,
    model=MODEL,
)

### Processamento de uma questão

In [ ]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

In [ ]:
assistant.total_cost()

In [ ]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

In [ ]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

### Processamento de todas as questões

In [ ]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

Teste em uma só questão

In [ ]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

In [ ]:
assistant.reset_usage()

Teste a base como um todo com processamento paralelo

Poucos workers por causa do limite de requisições do free tier do Gemini (as chamadas já têm retry com backoff).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [ ]:
assistant.total_cost()

In [ ]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

### LLM como juiz

In [ ]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

Saída esperada pelo juiz

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [ ]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [ ]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

Helper

In [ ]:
from dotenv import load_dotenv
from google import genai
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
genai_client = genai.Client()

MODEL = "gemini-3.5-flash-lite"

Testar com um registro

In [ ]:
rec = answers[0]

In [ ]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [ ]:
eval_result, usage = llm_structured_retry(
    genai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
    model=MODEL,
)

eval_result

In [ ]:
calc_price(usage)

Generalização

In [ ]:
def evaluate_aqa(question, answer_orig, answer_llm, model=MODEL):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        genai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [ ]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

### Executar o juiz

In [ ]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, answers, judge_record)

In [ ]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [ ]:
df_eval = pd.DataFrame(evaluations)

In [ ]:
calc_total_price(usages)

In [ ]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

In [ ]:
df_eval[df_eval["score"] == "bad"].head()

In [ ]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

## Agent

In [ ]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
genai_client = genai.Client()

MODEL = "gemini-3.5-flash-lite"

In [ ]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [ ]:
from evaluation_utils import AgentRunner

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = AgentRunner(
    llm_client=genai_client,
    tools=[search],
    instructions=instructions,
    model=MODEL,
)

No curso o loop vem do `toyaikit` (`OpenAIResponsesRunner`). Aqui o `AgentRunner` do `evaluation_utils` faz o mesmo loop do módulo 1 com o Gemini: gera a declaração da ferramenta a partir da assinatura e da docstring de `search`, guarda o `Content` inteiro do modelo no histórico e devolve as respostas das funções num único `Content` com `role="user"`. O resultado traz `last_message`, `all_messages`, `tool_calls` e `cost`.

In [ ]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [ ]:
result.all_messages

### Trajetória

Para a trajetória, nesse caso serão consideradas só as chamadas de funções.

No Gemini, as chamadas ficam nas `parts` dos `Content` com `role="model"` (`part.function_call`), e os argumentos (`args`) já vêm como `dict`, diferente da OpenAI, em que cada chamada é um item `type == "function_call"` com `arguments` em string JSON. O `extract_tool_calls` trata os dois casos e devolve sempre `{"name": ..., "arguments": dict}`.

In [ ]:
from evaluation_utils import extract_tool_calls

In [ ]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

Recupera a resposta original

In [ ]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

Salva o fluxo A -> Q -> A' e a trajetória

In [ ]:
import json

agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": json.dumps(tool_calls),
    "cost": result.cost,
    "document": doc_id,
}

agent_result

### Processa múltiplas questões

In [ ]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": json.dumps(tool_calls),
        "cost": result.cost,
        "document": doc_id,
    }

    return answer_record

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=2) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

In [ ]:
df_agent = pd.DataFrame(agent_answers)

In [ ]:
df_agent["cost"].sum()

In [ ]:
df_agent.to_csv("data/agent-answers.csv", index=False)

### LLM como juiz

Definição da saída esperada

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [ ]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

Função do juiz

In [ ]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model=MODEL):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        genai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [ ]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

In [ ]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [ ]:
with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

In [ ]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [ ]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [ ]:
calc_total_price(usages)

In [ ]:
df_agent_eval["answer_score"].value_counts()

In [ ]:
df_agent_eval["trajectory_score"].value_counts()

In [ ]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)